In [1]:


import random
import torch
torch.cuda.set_device(0)
import cv2
import numpy as np
import sys
import os
from omegaconf import OmegaConf
import torchvision

sys.path.append('..')
from tools.run_infinity import *
import warnings
warnings.filterwarnings("ignore") 

device = 'cuda:0'

model_path = '/home/yichen-xie/src/Infinity/checkpoints/Infinity-ysvs80_cd69d00000_0/ar-gpt_model_latest.pth'
folder_name = os.path.dirname(model_path)
config_path = os.path.join(folder_name, "config.yaml")


assert os.path.exists(config_path), f"config does not exist at {config_path}"

config = OmegaConf.load(config_path)
vae_path='../weights/infinity_vae_d16.pth'
text_encoder_ckpt = '../weights/flan-t5-xl'
model_type = 'infinitystream_layer12'

# vae_path='../weights/infinity_vae_d32reg.pth'
# text_encoder_ckpt = '../weights/flan-t5-xl'
# model_type = 'infinitystream_2b'

num_views = config.num_views
timesteps = config.timesteps
object_condition = config.object_condition
class_names = config.class_names
flash = config.flash

map_names = ['road_line_solid_single_white', 'road_line_broken_single_white', 'lane_surface_street', 'road_edge_sidewalk', 'road_edge_boundary', 'lane_surface_unstructure', 'crosswalk']

num_frames = 4
slides = 3
num_slides = (num_frames - timesteps) // slides

/home/yichen-xie/miniconda3/envs/inf_stream/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
2026-03-22 16:56:39.016262: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-03-22 16:56:39.794759: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/yichen-xie/miniconda3/envs/inf_stream/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
import lance
import scenarionet_tools.storage as storage
from scenarionet_tools.nuplan_data_wrapper import NuPlanDataWrapper, convert_bbox
from scenarionet_tools.map_utils import convert_points
from nuscenes_tools.nuscenes_utils.pipelines import Compose
import nuscenes_tools.nuscenes_utils.pipelines as pipelines
from nuscenes_tools.nuscenes_utils.box3d_instance import LiDARInstance3DBoxes
from infinity.utils.misc import get_scene_description, project_corners_to_views
from infinity.utils.vis_utils import draw_box_on_imgs, draw_map_points_on_imgs

os.environ['AWS_ACCESS_KEY_ID'] = 'cd0146b0fd5c24625a928b242d19f7e0dec18424'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'HN7mDT0pooo+3E40lyab8rrNfIied/33pCbpyrSEDuA='
os.environ['AWS_ENDPOINT_URL'] = 'https://idskhu5vqvtl.compat.objectstorage.us-phoenix-1.oraclecloud.com'
os.environ['AWS_DEFAULT_REGION'] = 'us-phoenix-1'
os.environ['PREAUTH_URL'] = 'https://idskhu5vqvtl.objectstorage.us-phoenix-1.oci.customer-oci.com/p/ofkGTeRQaWyr0mNvkheVidOQYGEjr4OmEhEAi3EECl_UjuMeqtvu8mKr-k22ixWw/n/idskhu5vqvtl/b/research_datasets/o/remote_deps'


dataset_path = 's3://research-datasets/unified_datasets/scenarionet_lite_nuplan_full_2025_04_29_lang_desc/'
# dataset_path = 's3://research-datasets/unified_datasets/scenarionet_lite_nuscenes_infinity/train'
mean=[0.5, 0.5, 0.5]
std=[0.5, 0.5, 0.5]
img_h, img_w = 192, 336
data_config={
    'dataset_path': dataset_path,
    'horizon': 16,
    'interval': 5,
    'random_interval': 1,
    
    'load_bbox': object_condition,
    'object_range': (-50, -50, -10, 50, 50, 10),

    'cams': ['CAM_F0', 'CAM_R0', 'CAM_R1', 'CAM_R2', 'CAM_B0', 'CAM_L2', 'CAM_L1', 'CAM_L0'],
    'Ncams': 8,
    #TODO: support higher resolution and bigger model
    'input_size': (img_h, img_w),
    'src_size': (1080, 1920),
    'keep_ratio': False,
    'mean': mean,
    'std': std,

    # Augmentation
    'resize': (0, 0),
    'rot': (0, 0),
    'flip': False,
    'crop_h': (0.0, 0.0),
    'resize_test':0.0,
    
    'load_map': True,
    'map_sample_points_num': 20,
}

lance_dataset = lance.dataset(
    dataset_path, storage_options=storage.make_s3_storage_options_from_env()
)

# build augmentations
if object_condition:
    eval_pipeline = [
        pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=True),
        pipelines.DefaultFormatBundle3D(class_names=class_names, map_names=map_names, with_gt=True, with_label=True),
        pipelines.Collect3D(keys=['img_inputs', 'gt_bboxes_3d', 'gt_labels_3d', 'map_sampled_points', 'map_type_labels']),
    ]
else:
    eval_pipeline = [
        pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=True),
        pipelines.DefaultFormatBundle3D(class_names=None, with_gt=False, with_label=False),
        pipelines.Collect3D(keys=['img_inputs']),
    ]

# build dataset
nuplan_dataset = NuPlanDataWrapper(data_config, 48, 49, eval_pipeline, seed=0)



In [3]:
def visualize_box(sample, video_imgs):
    all_bbox_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas'].data
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas'].data
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }


        camera_meta_data_list.append(camera_meta_datas)       

    bbox_sequence = []
    for tid in range(len(sample)):
        frame_data = sample[tid]
        gt_bboxes_3d = frame_data['gt_bboxes_3d'].data
        gt_labels_3d = frame_data['gt_labels_3d'].data

        if len(gt_labels_3d) > 0:
            xyz = gt_bboxes_3d.bottom_center
            dims = gt_bboxes_3d.dims
            yaw = gt_bboxes_3d.yaw
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            xyz, yaw = convert_bbox(xyz, yaw, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
            boxes = torch.cat([xyz, dims, yaw[:, None]], dim=-1)
            gt_bboxes_3d = LiDARInstance3DBoxes(boxes)
            bbox_num = boxes.shape[0]
        else:
            gt_bboxes_3d = LiDARInstance3DBoxes(torch.zeros((0, 7)))
            gt_labels_3d = torch.zeros(0)
            bbox_num = 0
        
        gt_labels_3d = gt_labels_3d.to(torch.int32)
        bbox_sequence.append([gt_bboxes_3d, gt_labels_3d, bbox_num])

    for tid in range(len(sample)):
        frame_meta_data = camera_meta_data_list[tid]
        frame_bbox = bbox_sequence[tid]
        
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [Image.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'gt_bboxes_3d': bbox_sequence[tid][0],
            'gt_labels_3d': bbox_sequence[tid][1],
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        bbox_imgs = draw_box_on_imgs(frame_data, frame_images, np.array(class_names))
        
        bbox_imgs = [torch.from_numpy(np.array(img)) for img in bbox_imgs]
        bbox_imgs = torch.stack(bbox_imgs, dim=0)
        bbox_imgs = bbox_imgs.permute(0, 3, 1, 2)
        all_bbox_imgs.append(bbox_imgs)
    
    return all_bbox_imgs

In [4]:
def save_imgs(images, suffix=''):
    TV3HW = torch.stack(images, dim=0)
    num_views = TV3HW.shape[1]
    TV3HW = TV3HW.flatten(0, 1)
    vthw = torchvision.utils.make_grid(TV3HW, nrow=num_views, padding=0, pad_value=1.0)
    vthw = vthw.clone().permute(1, 2, 0).cpu().numpy()
    vthw = PImage.fromarray(vthw.astype(np.uint8))
    vthw.save(f"../vis/debug/output_video_{i}{suffix}.png")

    # # Initialize video writer
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'mp4v' for MP4 format
    # video = cv2.VideoWriter(f"../debug/output_image_temporal_video_{i}.mp4", fourcc, 4, (img_w*num_views, img_h))

    # # Add images to the video
    # for image in video_imgs:
    #     frame = torch.unbind(image, dim=0)
    #     frame = torch.cat(frame, dim=2)
    #     frame = frame.permute(1, 2, 0).cpu().numpy()
    #     print(frame.shape, (img_w*num_views, img_h))
    #     frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    #     video.write(frame)

    # # Release the video writer
    # video.release()
    
    gif_frames = []
    for image in images:
        frame = torch.unbind(image, dim=0)
        frame = torch.cat(frame, dim=2)
        frame = frame.permute(1, 2, 0).cpu().numpy()
        gif_frames.append(Image.fromarray(frame))
    gif_frames[0].save(f"../vis/debug/output_video_{i}{suffix}.gif", save_all=True, append_images=gif_frames[1:], duration=500, loop=0)



In [5]:
def get_gt_images(sample):
    gt_images = []
    for tid in range(len(sample)):
        frame_sample = sample[tid]
        frame_images = frame_sample['img_inputs'][0].clone()

        frame_images = (frame_images + 1) / 2
        frame_images = (frame_images*255).to(torch.uint8)
        
        gt_images.append(frame_images)
    return gt_images

In [6]:
def visualize_map_points(sample, video_imgs):
    all_map_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas'].data
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas'].data
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }


        camera_meta_data_list.append(camera_meta_datas)       

    for tid in range(len(sample)):
        frame_data = sample[tid]
        map_points = frame_data['map_sampled_points'].data.to(torch.float32)
        map_labels = frame_data['map_type_labels'].data
        
        if len(map_labels) > 0:
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            frame_points = convert_points(map_points, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
        else:
            frame_points = None
                
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [Image.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'map_type_labels': map_labels,
            'map_sampled_points': frame_points,
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        map_imgs = draw_map_points_on_imgs(frame_data, frame_images, np.array(map_names))
        
        map_imgs = [torch.from_numpy(np.array(img)) for img in map_imgs]
        map_imgs = torch.stack(map_imgs, dim=0)
        map_imgs = map_imgs.permute(0, 3, 1, 2)
        all_map_imgs.append(map_imgs)
    
    return all_map_imgs

In [7]:
import cv2
from PIL import Image

nuplan_iterator = iter(nuplan_dataset)

for i in range(0, 10):
    print(i)
    sample = next(nuplan_iterator)
    gt_images = get_gt_images(sample)
    
    gt_images_bbox = visualize_box(sample, gt_images)
    gt_images_map = visualize_map_points(sample, gt_images_bbox)

    save_imgs(gt_images_map, '_gt')
    

0
1
2


KeyboardInterrupt: 

In [ ]:
# print(sample[0]['map_type_labels'].data.shape)
# print(sample[0]['map_sampled_points'].data)